# ♟️ GMAI — Training Analysis

Loads the CSV log written by `gmai.train` and inspects the learning dynamics:
episode reward, rolling win-rate, exploration (ε) and TD loss.

> **Como ler:** os gráficos abaixo só têm significado depois de um treino real
> (`python -m gmai.train --config configs/default.yaml`). Com poucos episódios,
> a win-rate ainda reflete sobretudo a sorte contra o oponente aleatório.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Point this at your run directory
RUN_DIR = sorted(Path("../runs").glob("*"))[-1]  # latest run
df = pd.read_csv(RUN_DIR / "logs.csv")
df["loss"] = pd.to_numeric(df["loss"], errors="coerce")
print(f"run: {RUN_DIR.name} | episodes: {len(df)} | stages: {df['stage'].unique().tolist()}")
df.tail()

In [ ]:
WINDOW = 100

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle("GMAI training dynamics", fontsize=14)

ax = axes[0, 0]
ax.plot(df["episode"], df["reward"], alpha=0.25, lw=0.7, color="#66bb6a")
ax.plot(df["episode"], df["reward"].rolling(WINDOW).mean(), color="#1b5e20", lw=2)
ax.set_title("Episode reward (rolling mean)")
ax.set_xlabel("episode"); ax.set_ylabel("reward")

ax = axes[0, 1]
ax.plot(df["episode"], df["result"].rolling(WINDOW).mean(), color="#2e7d32", lw=2)
ax.axhline(0.5, ls="--", c="grey", lw=1)
# mark curriculum stage transitions
transitions = df.groupby("stage")["episode"].min().sort_values()
for stage, ep in transitions.items():
    if ep > 1:
        ax.axvline(ep, color="#9e9e9e", ls=":", lw=1)
        ax.text(ep, 0.05, stage, rotation=90, fontsize=8, color="#616161")
ax.set_title(f"Win-rate (rolling {WINDOW})")
ax.set_xlabel("episode"); ax.set_ylabel("score"); ax.set_ylim(0, 1)

ax = axes[1, 0]
ax.plot(df["episode"], df["epsilon"], color="#43a047", lw=2)
ax.set_title("Exploration ε"); ax.set_xlabel("episode")

ax = axes[1, 1]
ax.plot(df["episode"], df["loss"], alpha=0.3, lw=0.7, color="#66bb6a")
ax.plot(df["episode"], df["loss"].rolling(WINDOW).mean(), color="#1b5e20", lw=2)
ax.set_yscale("log")
ax.set_title("TD loss (log scale)"); ax.set_xlabel("episode")

plt.tight_layout()
plt.show()

**Como ler**

- **Reward** sobe primeiro por efeito do *shaping* de material (o agente aprende a capturar), só depois pelas vitórias em si — é o comportamento esperado de shaping potential-based.
- **Win-rate**: as linhas verticais marcam as promoções do currículo. É normal a win-rate **cair** a cada promoção (o adversário ficou mais forte) e recuperar depois — o padrão em dente de serra é o currículo a funcionar, não um bug.
- **ε** deve atingir o piso (0.05) bem antes do fim do treino; se ainda estiver alto, aumenta `epsilon.decay_steps`.
- **TD loss** em escala log: picos após cada sync da target network são normais; tendência crescente sustentada sugere `lr` alto demais.

In [ ]:
# Arena check against the fixed baselines (100 games each)
from gmai.agent import DQNAgent
from gmai.evaluate import run_arena

agent = DQNAgent()
agent.load(RUN_DIR / "final.pt")
agent.epsilon = 0.0

report = run_arena(agent, games=100, seed=0)
pd.DataFrame(report).T